In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root      51300 F...m python3
/dev/nvidiactl:      root      51300 F...m python3
/dev/nvidia-uvm:     root      51300 F...m python3


# Libraries

In [2]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==5.17.0" \
    "peft==0.21.0" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [3]:
import json
import math
import torch
from datetime import datetime
from pathlib import Path
from collections import OrderedDict

from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from huggingface_hub import snapshot_download
from safetensors.torch import load_file

# Configurations

In [4]:
# Run configuration
SRC_LANG = 'en' # 'en' | '<any_lang>'
TGT_LANG = 'tr' # 'en' | '<any_lang>'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
# ---
# LegameX-Void runs (FILL IN after the Void training runs finish; the run name
# format matches legamex_void_training_xlmr.ipynb, e.g.
#   '<username>/XLM-R-Base-squad-en-15K-s42-LegameX-Void-LoRA-v<timestamp>')
LEGAMEX_TASK_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LegameX-Void-LoRA-v261121152724'
LEGAMEX_TASK_CKPT_STEP = 6000
LEGAMEX_LANG_ID = 'alxxtexxr/XLM-R-Base-wikipedia-tr-3.5K-s42-LegameX-Void-LoRA-v261122104103'
LEGAMEX_LANG_CKPT_STEP = 800
# ---

REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-2K-s42-LoRA-v260929201303'
REF_LORA_CKPT_STEP = 1000
# REF_LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-tr-3.5K-s42-LoRA-v261020114224'
# REF_LORA_CKPT_STEP = 400
REF_DELTA_WEIGHT = 1.0  # weight of the reference-LoRA delta (used by 'awr' only)

# Merging configuration.
# LegameX-Void has NO gate, so each model contributes its RAW transfer delta at
# a plain scalar weight (no gate-derived coefficients):
#   W_merged = W_base + SUM_i c_i * d_i,  c_i = addition_weights[i]
#   'add': c_i = 1.0      W = W_base + d_task     + d_lang
#                          (raw transfer deltas at full strength; ref omitted)
#   'avg': c_i = 0.5      W = W_base + 0.5*d_task + 0.5*d_lang
#                          (scalar-weighted average of transfer deltas; ref omitted)
#   'awr': c_i = 1.0      'add' PLUS the shared reference-LoRA delta:
#                          W = W_base + d_task + d_lang + REF_DELTA_WEIGHT * d_ref
# The reference LoRA (ref) is the SAME pretrained component in every Void run
# (base + ref + tfr per layer), so it is not reconstructed from the checkpoints:
# 'awr' adds it ONCE from the hub adapter. REF_DELTA_WEIGHT = 1.0 reproduces the
# full Void composition (base + ref + d_task + d_lang).
ADDITION_TYPE = 'avg'  # 'add' | 'avg' | 'awr'

assert ADDITION_TYPE in ['add', 'avg', 'awr'],     f"Invalid addition type: {ADDITION_TYPE}. Must be 'add', 'avg', or 'awr'."
if ADDITION_TYPE == 'add':
    addition_weights = [1.0, 1.0]
    # addition_weights = [1.0, 0.0]
elif ADDITION_TYPE == 'avg':
    addition_weights = [0.5, 0.5]
elif ADDITION_TYPE == 'awr':
    # The Void part is 'add' (weights [1,1] -> raw deltas at full strength); the
    # shared reference-LoRA delta is added on top at REF_DELTA_WEIGHT in the
    # merge cell.
    addition_weights = [1.0, 1.0]
    # addition_weights = [1.0, 0.0]

# Determine the hub merged model ID
assert SRC_LANG in LEGAMEX_TASK_ID and TGT_LANG in LEGAMEX_LANG_ID, \
    "LegameX IDs do not match the specified source language and target language."
model_id_0, model_id_1 = LEGAMEX_TASK_ID.split(SRC_LANG)
model_id_1_0 = model_id_1.split("K-")[-1].split("-v")[0]
type_tag = ADDITION_TYPE
hub_merged_model_id = f"{model_id_0}{TGT_LANG}-{model_id_1_0}-{type_tag}-v{datetime.now().strftime('%y%m%d%H%M%S')}"
print(f"Hub merged model ID: {hub_merged_model_id}")


Hub merged model ID: alxxtexxr/XLM-R-Base-squad-tr-s42-LegameX-Void-LoRA-avg-v261122140531


# Utilities

In [5]:
# LoRA utilities
def download_hf_model(
        repo_id: str, 
        ckpt_step: int, 
        max_checkpoints: int = 10_000,
        ckpt_interval: int = 25,
    ) -> tuple[Path, Path]:
    assert len(repo_id.split('/')) == 2, \
        f"Invalid repo_id: {repo_id}. Must be in the format 'username/repo_name'."
    local_dir = Path(repo_id.split('/')[-1])
    ignore_checkpoints = None
    
    if ckpt_step is not None:
        ignore_checkpoints = [
            f'checkpoint-{i}/*' for i in range(0, max_checkpoints, ckpt_interval) 
            if i != ckpt_step
        ]
        ignore_checkpoints += ['runs/*', '**/wandb/*']

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        ignore_patterns=ignore_checkpoints,
    )

    ckpt_dir = None
    if ckpt_step is not None:
        ckpt_dir = local_dir / f'checkpoint-{ckpt_step}'
    return local_dir, ckpt_dir

# LoRA scaling / delta utilities.
# These mirror the scaling math in notebooks/legamex_void_training_xlmr.ipynb
# (LegamexVoidLora._init_lora_module) so that a merge reconstructs the exact
# transfer delta that was applied during training.

def tfr_lora_scaling(legamex_config: dict) -> float:
    """Transfer-LoRA scaling alpha / r (alpha / sqrt(r) when use_rslora).
    Mirrors LegamexVoidLora._init_lora_module's scaling."""
    tfr_lora_config = legamex_config['tfr_lora_config']
    r = tfr_lora_config['r']
    alpha = tfr_lora_config['lora_alpha']
    return alpha / math.sqrt(r) if tfr_lora_config.get('use_rslora', False) else alpha / r


def fix_lora_state_dict(
    lora_state_dict: dict[str, torch.Tensor],
    adapter_config: dict,
) -> OrderedDict[str, torch.Tensor]:
    """Convert a standard PEFT LoRA adapter into raw deltas keyed by base-model
    param names: {<module>.weight: scaling * (B @ A)}.

    The reference LoRA is a plain (ungated) LoRA, so its contribution to a merge
    is simply its delta. Keys are stripped of the 'base_model.model.' prefix so
    they align with AutoModelForQuestionAnswering param names.
    """
    r = adapter_config['r']
    alpha = adapter_config['lora_alpha']
    scaling = alpha / math.sqrt(r) if adapter_config.get('use_rslora', False) else alpha / r

    lora_A: dict[str, torch.Tensor] = {}
    lora_B: dict[str, torch.Tensor] = {}
    for key, value in lora_state_dict.items():
        if key.endswith('.lora_A.weight'):
            lora_A[key[: -len('.lora_A.weight')]] = value
        elif key.endswith('.lora_B.weight'):
            lora_B[key[: -len('.lora_B.weight')]] = value

    assert lora_A and lora_A.keys() == lora_B.keys(), \
        "Reference LoRA must have matching lora_A/lora_B weights."

    delta_state_dict = OrderedDict()
    for module, B in lora_B.items():
        module_name = module.removeprefix('base_model.model.')
        delta_state_dict[f'{module_name}.weight'] = scaling * (B @ lora_A[module])
    return delta_state_dict


# Model

In [6]:
_, legamex_lang_dir = download_hf_model(repo_id=LEGAMEX_LANG_ID, ckpt_step=LEGAMEX_LANG_CKPT_STEP)
legamex_lang_file = legamex_lang_dir / 'legamex.pt'
legamex_lang_state_dict = torch.load(legamex_lang_file, map_location='cpu')
# legamex_config.json holds the merge-time constants (tfr lora_alpha/r) that
# are NOT stored in legamex.pt
with open(legamex_lang_dir / 'legamex_config.json', 'r') as f:
    legamex_lang_config = json.load(f)

_, legamex_task_dir = download_hf_model(repo_id=LEGAMEX_TASK_ID, ckpt_step=LEGAMEX_TASK_CKPT_STEP)
legamex_task_file = legamex_task_dir / 'legamex.pt'
legamex_task_state_dict = torch.load(legamex_task_file, map_location='cpu')
with open(legamex_task_dir / 'legamex_config.json', 'r') as f:
    legamex_task_config = json.load(f)

_, lora_ref_dir = download_hf_model(repo_id=REF_LORA_ID, ckpt_step=REF_LORA_CKPT_STEP)
lora_ref_file = lora_ref_dir / 'adapter_model.safetensors'
lora_ref_state_dict = load_file(lora_ref_file, device='cpu')
with open(lora_ref_dir / 'adapter_config.json', 'r') as f:
    lora_ref_config = json.load(f)

# Raw reference-LoRA deltas keyed by base-model param names (only used by 'awr')
lora_ref_delta_state_dict = fix_lora_state_dict(lora_ref_state_dict, lora_ref_config)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

In [7]:
def fix_legamex_state_dict(
    state_dict: dict[str, torch.Tensor],
    legamex_config: dict,
) -> OrderedDict[str, torch.Tensor]:
    """Reconstruct each LegameX-Void module's RAW transfer delta.

    LegameX-Void has no gate: the layer delta is the plain sum of the reference
    (ref) and transfer (tfr) LoRA deltas (base + ref_delta + tfr_delta). The
    reference LoRA is a SHARED pretrained component (identical weights in every
    Void run), so it is NOT reconstructed here - it is added once from the hub
    adapter in the 'awr' mode. What each model contributes uniquely is its
    transfer delta:

        d_i = tfr_scaling * (B_tfr @ A_tfr)

    mirroring LegamexVoidLora's forward in legamex_void_training_xlmr.ipynb.

    qa_outputs entries are passed through as FULL trained head weights, not deltas.

    Returns:
        delta_state_dict: per-module RAW transfer delta d_i, plus qa_outputs.
    """
    tfr_scaling = tfr_lora_scaling(legamex_config)
    print(f"tfr_scaling={tfr_scaling}")

    delta_state_dict = OrderedDict()

    lora_tfr_A_weight = None
    lora_tfr_B_weight = None

    for key, value in state_dict.items():
        key_fixed = key.replace('__DOT__', '.')

        if 'lora.tfr.A' in key_fixed:
            lora_tfr_A_weight = value
        elif 'lora.tfr.B' in key_fixed:
            lora_tfr_B_weight = value
        elif 'qa_outputs' in key_fixed:
            delta_state_dict[key_fixed] = value
            continue
        else:
            # lora.ref.* (the shared reference) is intentionally skipped
            continue

        if lora_tfr_A_weight is not None and lora_tfr_B_weight is not None:
            # RAW transfer delta d_i WITH the LoRA scaling factor (alpha / r)
            delta_weight = (lora_tfr_B_weight @ lora_tfr_A_weight) * tfr_scaling

            key_base = key_fixed.split('.lora.')[0] + '.weight'
            delta_state_dict[key_base] = delta_weight

            lora_tfr_A_weight = None
            lora_tfr_B_weight = None

    return delta_state_dict


legamex_task_delta = fix_legamex_state_dict(
    legamex_task_state_dict, legamex_task_config)
legamex_lang_delta = fix_legamex_state_dict(
    legamex_lang_state_dict, legamex_lang_config)


tfr_scaling=1.0
tfr_scaling=1.0


In [8]:
# LegameX-Void has no gates, so the merge coefficient of each model's RAW
# transfer delta is a plain scalar (addition_weights from the config cell):
#   'add': c_i = 1.0        W = W_base + d_task     + d_lang
#   'avg': c_i = 0.5        W = W_base + 0.5*d_task + 0.5*d_lang
#   'awr': c_i = 1.0        'add' + REF_DELTA_WEIGHT * d_ref (added in the merge cell)
# ORDERING: fixed_state_dicts in the merge cell is [task, lang]; addition_weights
# is indexed the same ([0] = task, [1] = lang).
assert addition_weights is not None and len(addition_weights) == 2,     f"'add'/'avg'/'awr' need two scalar weights, got {addition_weights!r}"
print(f"[{ADDITION_TYPE}] transfer delta weights (task, lang): {addition_weights}")


[avg] transfer delta weights (task, lang): [0.5, 0.5]


In [9]:
# Load the base model
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForQuestionAnswering LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
qa_outputs.weight           | MISSING    | 
qa_outputs.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Merge with Weighted Addition / Averaging


In [10]:
# Weighted merge over the models' RAW transfer deltas.
#
#   W_merged = W_base + SUM_i c_i * d_i
#
# where d_i is each model's RAW transfer delta (from fix_legamex_state_dict) and
# c_i is its scalar merge coefficient (addition_weights from the config cell):
#   'add': c_i = 1.0   raw deltas at full strength
#   'avg': c_i = 0.5   scalar-weighted average
# This mirrors the sequential PEFT merge in notebooks/lora_addition_xlmr.ipynb
# (W <- W + w * (B @ (w-scaled) A)), but with the deltas precomputed.
#
# Ordering: fixed_state_dicts and addition_weights are both [task, lang], so
# zip() pairs each delta with its own coefficient.
fixed_state_dicts = [
    legamex_task_delta,
    legamex_lang_delta,
]

# Defensive normalization: strip a 'base_model.' prefix if present
# (current checkpoints do not use one; older versions did).
fixed_state_dicts = [
    {k.removeprefix('base_model.'): v for k, v in state_dict.items()}
    for state_dict in fixed_state_dicts
]

current_model = base_model
for delta_state_dict, coeff in zip(fixed_state_dicts, addition_weights):
    n_deltas, n_heads = 0, 0
    with torch.no_grad():
        for name, param in current_model.named_parameters():
            if name not in delta_state_dict:
                continue
            value = delta_state_dict[name].to(device=param.device, dtype=param.dtype)
            if 'qa_outputs' in name:
                # qa_outputs entries are FULL trained weights, not deltas:
                # the base XLM-R QA head is randomly initialized, so we copy
                # the trained head instead of adding it on top of noise.
                # (Only the squad/task LegameX provides a QA head; the
                #  wikipedia lang LegameX is FEATURE_EXTRACTION and has none.)
                param.data.copy_(value)
                n_heads += 1
            else:
                # W += c_i * d_i   (c_i is the scalar merge coefficient)
                param.data += coeff * value
                n_deltas += 1
    print(f"[{ADDITION_TYPE}] model: added {n_deltas} deltas (coefficient c_i={coeff}), copied {n_heads} qa_outputs tensors")

# 'awr': add the shared reference-LoRA delta on top of the transfer deltas.
# The reference LoRA is a plain (ungated) LoRA - the SAME pretrained reference
# used by every LegameX-Void run - so its contribution is just its delta scaled
# by REF_DELTA_WEIGHT, added ONCE. It targets only attention/FFN modules (NOT
# qa_outputs), so the task model's trained QA head is preserved.
if ADDITION_TYPE == 'awr':
    n_ref = 0
    with torch.no_grad():
        for name, param in current_model.named_parameters():
            if name not in lora_ref_delta_state_dict:
                continue
            value = lora_ref_delta_state_dict[name].to(device=param.device, dtype=param.dtype)
            param.data += REF_DELTA_WEIGHT * value
            n_ref += 1
    print(f"[awr] added {n_ref} reference-LoRA deltas at REF_DELTA_WEIGHT={REF_DELTA_WEIGHT}")

merged_model = current_model


[avg] model: added 72 deltas (coefficient c_i=0.5), copied 2 qa_outputs tensors
[avg] model: added 72 deltas (coefficient c_i=0.5), copied 0 qa_outputs tensors


In [11]:
# Sanity check
for i in range(11):
    print(f"Merged model layer-{i} attention value weight norm:", merged_model.roberta.encoder.layer[i].attention.self.value.weight.norm().item())
print()
print("Merged model qa_outputs weight norm:", merged_model.qa_outputs.weight.norm().item())
print("Merged model qa_outputs bias norm:", merged_model.qa_outputs.bias.norm().item())

# 'add' / 'avg' / 'awr': report the scalar coefficients actually applied
print()
print(f"[{ADDITION_TYPE}] transfer delta coefficients (task, lang): {addition_weights}")

# 'awr': report the reference-LoRA delta coverage (should exclude qa_outputs)
if ADDITION_TYPE == 'awr':
    ref_keys = list(lora_ref_delta_state_dict.keys())
    print()
    print(f"'awr' reference-LoRA deltas: {len(ref_keys)} tensors, "
          f"qa_outputs present: {any('qa_outputs' in k for k in ref_keys)}")
    print(f"e.g. {ref_keys[0]}")


Merged model layer-0 attention value weight norm: 28.36480712890625
Merged model layer-1 attention value weight norm: 28.846416473388672
Merged model layer-2 attention value weight norm: 30.103307723999023
Merged model layer-3 attention value weight norm: 36.345394134521484
Merged model layer-4 attention value weight norm: 38.295433044433594
Merged model layer-5 attention value weight norm: 41.630855560302734
Merged model layer-6 attention value weight norm: 38.20491409301758
Merged model layer-7 attention value weight norm: 38.961490631103516
Merged model layer-8 attention value weight norm: 39.16530990600586
Merged model layer-9 attention value weight norm: 34.903472900390625
Merged model layer-10 attention value weight norm: 31.03420066833496

Merged model qa_outputs weight norm: 1.215196132659912
Merged model qa_outputs bias norm: 0.004364593420177698

[avg] transfer delta coefficients (task, lang): [0.5, 0.5]


In [12]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...25w0pzr/model.safetensors:   1%|          | 7.98MB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mprvyt2lz0/tokenizer.json:  93%|#########3| 15.9MB / 17.1MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/XLM-R-Base-squad-tr-s42-LegameX-Void-LoRA-avg-v261122140531
